<a href="https://colab.research.google.com/github/lidiaflores/classify-diagnosis/blob/main/embedding/embedding_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import numpy as np

from sentence_transformers import SentenceTransformer

import pandas as pd
from datasets import load_dataset

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import label_binarize
from sklearn.metrics import accuracy_score, auc, classification_report, roc_auc_score, f1_score, confusion_matrix, roc_curve,precision_score, recall_score, log_loss
from sklearn.calibration import calibration_curve

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data = load_dataset("gretelai/symptom_to_diagnosis") #hugging face dataset
train_symptoms, train_label = list(data["train"]["input_text"]), list(data["train"]["output_text"])
test_symptoms, test_label = list(data["test"]["input_text"]), list(data["test"]["output_text"])
train_df = pd.DataFrame(data["train"])
test_df = pd.DataFrame(data["test"])

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


README.md:   0%|          | 0.00/2.46k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/171k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/42.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/853 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/212 [00:00<?, ? examples/s]

Train shape: (853, 2)
Test shape: (212, 2)


In [3]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")

train_emb = encoder.encode(train_symptoms, show_progress_bar=True)
test_emb = encoder.encode(test_symptoms, show_progress_bar=True)

print("train set embeddings:", train_emb.shape)
print("test set embeddings:", test_emb.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

train set embeddings: (853, 384)
test set embeddings: (212, 384)


In [17]:
# 5 fold cv
crossval = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_model(model_name, model, X_train, X_test):

    cv = cross_validate(model, X_train, train_label, cv=crossval, scoring=["accuracy", "f1_weighted"])
    cv_accuracy, cv_f1 = cv["test_accuracy"], cv["test_f1_weighted"]

    print(model_name)
    print("CV accuracy at every fold:", cv_accuracy.round(3))
    print("CV accuracy:", round(cv_accuracy.mean(), 3), "std", round(cv_accuracy.std(), 3))
    print("CV f1 score:", round(cv_f1.mean(), 3), "std", round(cv_f1.std(), 3))
    model.fit(X_train, train_label)
    clf = model.predict(X_test)

    classes = model.classes_
    binary_matrix = label_binarize(test_label, classes=classes)
    proba = model.predict_proba(X_test)

    auc_score = roc_auc_score(binary_matrix, proba, average='weighted', multi_class='ovr')
    f1 = f1_score(test_label, clf, average='weighted')
    precision = precision_score(test_label, clf, average='weighted', zero_division=0)
    recall = recall_score(test_label, clf, average='weighted')

    print("accuracy:", accuracy_score(test_label, clf))
    print('f1 score:', f1, "AUC score", auc_score)

    brier= np.mean(np.sum((proba - binary_matrix) **2, axis=1))
    logloss= log_loss(test_label, proba, labels=classes)
    print('brier score:', brier, "logloss", logloss)

    train_accuracy = accuracy_score(train_label, model.predict(X_train))
    test_accuracy = accuracy_score(test_label, clf)
    print("Train accuracy:", train_accuracy)
    print("Test accuracy:", test_accuracy)
    print("Gap", train_accuracy - test_accuracy)
    print(classification_report(test_label, clf, zero_division=0))

    return {"model": model_name, "accuracy": test_accuracy, "precision": precision,
            "recall": recall, "f1": f1, "auc": auc_score, "brier": brier, "loss_loss": logloss}

In [18]:
#hyper param tuning
def tune_le_models(model,grid):
  search=GridSearchCV(model,grid, cv=crossval, scoring='f1_weighted',n_jobs=-1, verbose=1)
  search.fit(train_emb, train_label)
  print('best hyper params:', search.best_params_, "cv f1:", round(search.best_score_, 3))
  return search.best_estimator_

In [19]:
# call models
log_reg = tune_le_models(OneVsRestClassifier(LogisticRegression(solver="lbfgs", random_state=42, max_iter=1000)),{"estimator__C": [0.1, 1,10,100]})
svm = tune_le_models(SVC(kernel="linear", probability=True, random_state=42), {"C": [0.1, 1, 10]})
decision_tree = tune_le_models(DecisionTreeClassifier(random_state=42), {"max_depth": [5,10,20,None], "min_samples_leaf": [1, 3, 5]})
random_forest = tune_le_models(RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1), {"max_depth": [10,20,None], "max_features": ["sqrt", 0.3]})

results = [
    evaluate_model("Logistic Regression", log_reg, train_emb, test_emb),
    evaluate_model("Linear SVM", svm, train_emb, test_emb),
    evaluate_model("Decision Tree", decision_tree, train_emb, test_emb),
    evaluate_model("Random Forest", random_forest, train_emb, test_emb),
]

results_df = pd.DataFrame(results)
print(results_df.round(3))

Fitting 5 folds for each of 4 candidates, totalling 20 fits
best hyper params: {'estimator__C': 100} cv f1: 0.956
Fitting 5 folds for each of 3 candidates, totalling 15 fits
best hyper params: {'C': 10} cv f1: 0.966
Fitting 5 folds for each of 12 candidates, totalling 60 fits
best hyper params: {'max_depth': 20, 'min_samples_leaf': 1} cv f1: 0.61
Fitting 5 folds for each of 6 candidates, totalling 30 fits
best hyper params: {'max_depth': None, 'max_features': 'sqrt'} cv f1: 0.908
Logistic Regression
CV accuracy at every fold: [0.953 0.959 0.965 0.971 0.935]
CV accuracy: 0.957 std 0.012
CV f1 score: 0.956 std 0.013
accuracy: 0.9339622641509434
f1 score: 0.9328544724741931 AUC score 0.9982501410286023
brier score: 0.1081501740330994 logloss 0.2232503973789241
Train accuracy: 1.0
Test accuracy: 0.9339622641509434
Gap 0.06603773584905659
                                 precision    recall  f1-score   support

                        allergy       0.90      0.90      0.90        10
       

In [20]:
#bootstrap w replacement on test
def bootstrap_ci(model, n_boot=1000):
    rng = np.random.default_rng(42)
    y = np.array(test_label)
    pred = model.predict(test_emb)

    scores = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(np.where(y == c)[0], size=(y == c).sum(), replace=True)
                              for c in np.unique(y)])
        scores.append(accuracy_score(y[idx], pred[idx]))

    low, high = np.percentile(scores, [2.5, 97.5])
    return low, high


for name, model in [("Logistic Regression", log_reg), ("Linear SVM", svm),
                    ("Decision Tree", decision_tree), ("Random Forest", random_forest)]:
    low, high = bootstrap_ci(model)
    print(f"{name}: accuracy 95% CI {low:.3f} - {high:.3f}")

Logistic Regression: accuracy 95% CI 0.901 - 0.962
Linear SVM: accuracy 95% CI 0.915 - 0.972
Decision Tree: accuracy 95% CI 0.571 - 0.675
Random Forest: accuracy 95% CI 0.873 - 0.939
